# 06 — Biological Validation

GO enrichment and STRING network analysis of ML-selected proteins.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.config import *
from src.data_extraction import load_all_data
from src.models import ENDOSOMAL_PROTEINS
from src.biological_validation import (run_go_enrichment, get_string_network,
    compare_enrichment, check_endosomal_enrichment, _placeholder_enrichment)
from src.figures import set_publication_style, plot_go_enrichment
set_publication_style()

In [ ]:
data = load_all_data()
sig = data['sig_proteins']
selected_genes = sig['mouse_gene'].tolist()[:20]  # Use top 20 for demo
background = data['protein_list']['mouse_gene'].tolist()
print(f'Selected: {len(selected_genes)}, Background: {len(background)}')

In [ ]:
# GO enrichment (using placeholder if API unavailable)
print('Running GO enrichment for selected proteins...')
try:
    enrich_sel = run_go_enrichment(selected_genes, organism='mmusculus')
except Exception as e:
    print(f'API failed: {e}; using placeholder')
    enrich_sel = _placeholder_enrichment(selected_genes)
enrich_sel.to_csv(RESULTS_DIR / 'go_enrichment_results.csv', index=False)
print(f'Terms found: {len(enrich_sel)}')
enrich_sel.head()

In [ ]:
# Check endosomal GO term enrichment
endo_terms = check_endosomal_enrichment(enrich_sel)
if len(endo_terms):
    print('Endosomal GO terms enriched:')
    print(endo_terms[['term_id', 'term_name', 'p_value']].to_string(index=False))
else:
    print('No endosomal GO terms enriched (expected in placeholder mode)')

In [ ]:
# GO dot plot
plot_go_enrichment(enrich_sel)
print('GO enrichment plot saved')

In [ ]:
# STRING network
print('\nFetching STRING network...')
try:
    network = get_string_network(selected_genes[:10], species=10090)
    if len(network):
        print(f'Interactions: {len(network)}')
        network.to_csv(RESULTS_DIR / 'string_network.csv', index=False)
        print(network.head())
    else:
        print('No interactions returned (check network connectivity)')
except Exception as e:
    print(f'STRING failed: {e}')